<h1 style="text-align: center;">[Prediksi Customer Churn pada Perusahaan Telco]</h1>
<h3 style="text-align: center;">[Firsa Adam, Yoanita Dwi Harlandi Dan Andre Yonathan]</h3>

---

## **Section 0. Setup**

>   *Tujuan:* Menyiapkan environment kerja (import library, konfigurasi global) supaya proses selanjutnya konsisten dan reproducible.

**0.1 Import Library**

>   *Tujuan:* Memuat semua library yang dibutuhkan di satu tempat di awal, supaya dependency notebook mudah dilacak.

In [2]:
# import library dasar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# tambahkan import lain sesuai kebutuhan (sklearn, dll) di sini

**0.2 Global Configuration**

>   *Tujuan:* Menetapkan parameter global seperti random_state di satu tempat, supaya semua proses (split, model, CV) memakai nilai yang sama.

> **Catatan** Selalu set `RANDOM_STATE` di satu tempat dan pakai variabel yang sama di semua proses (train_test_split, model, cross-validation). Ini membuat hasil eksperimenmu **reproducible** - orang lain (atau kamu sendiri, minggu depan) bisa menjalankan ulang notebook dan dapat hasil yang sama persis.

In [3]:
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
TARGET_COL = 'Churn'                          # kolom target
POSITIVE_CLASS = 'Yes'

## **Section 1. Business Understanding**

>   *Tujuan:* Menerjemahkan masalah bisnis menjadi masalah yang bisa diselesaikan dengan machine learning, sebelum menyentuh data sama sekali.

**1.1 Context**

> *Tujuan:* Menjelaskan latar belakang bisnis: siapa stakeholder-nya dan kenapa masalah ini perlu diselesaikan sekarang.

Perusahaan telekomunikasi (telco) menjalankan model bisnis subscription-based, di mana pendapatan utama berasal dari pelanggan yang membayar layanan secara berkala setiap bulan. Dalam model bisnis seperti ini, mempertahankan pelanggan (retention) menjadi salah satu faktor utama untuk menjaga recurring revenue, karena setiap pelanggan yang keluar (churn) berarti hilangnya pendapatan berulang sekaligus menambah beban biaya akuisisi pelanggan baru untuk menggantikannya

Berdasarkan data historis (4.930 pelanggan), tercatat **churn rate sebesar 26,69%**, lebih tinggi dibandingkan benchmark rata-rata industri telekomunikasi yang berada di kisaran **21,5–22%** (https://focus-digital.co/average-churn-rate-by-industry/). Selisih ini menunjukkan adanya ruang perbaikan jika dibandingkan dengan kondisi industri secara umum.

Berdasarkan riset Harvard Business Review dan Frederick Reichheld (Bain & Company), biaya untuk memperoleh pelanggan baru diperkirakan 5–25 kali lebih tinggi dibandingkan biaya mempertahankan pelanggan yang sudah ada, tergantung industri. Karena itu, kemampuan mengidentifikasi pelanggan berisiko churn sejak dini menjadi krusial agar upaya retensi dapat dilakukan secara proaktif — bukan reaktif setelah pelanggan benar-benar berhenti.

Pola churn pelanggan pada dasarnya dipengaruhi oleh kombinasi banyak faktor (jenis kontrak, layanan tambahan yang digunakan, lama berlangganan, biaya bulanan, dan lainnya) yang sulit ditangkap secara akurat dengan aturan bisnis manual sederhana. Karena itu, dibutuhkan pendekatan machine learning yang mampu mempelajari pola dari data historis pelanggan.

Pihak yang berkepentingan langsung terhadap hasil project ini:

•	**Tim CRM** — menggunakan skor risiko churn untuk mengidentifikasi pelanggan berisiko tinggi dan menjalankan program retensi secara tersegmentasi (high/medium/low risk).

•	**Tim Marketing** — memanfaatkan daftar pelanggan berisiko untuk menjalankan kampanye retensi yang lebih tepat sasaran.

•	**Tim Product** — menggunakan feature importance dan insight model untuk mengevaluasi produk, kontrak, atau layanan yang paling berkontribusi terhadap churn.


**1.2 Problem Statements**

> *Tujuan:* Merumuskan masalah secara spesifik dan terukur, biasanya dalam bentuk pertanyaan yang bisa dijawab lewat data.

Berdasarkan konteks bisnis di atas, masalah dirumuskan menjadi pertanyaan yang dapat dijawab menggunakan data pelanggan yang tersedia:

1.	Pelanggan dengan karakteristik seperti apa (jenis kontrak, layanan tambahan, lama berlangganan, biaya bulanan, dsb.) yang memiliki risiko churn paling tinggi?
2.	Dapatkah dibangun model klasifikasi yang mampu memprediksi probabilitas seorang pelanggan akan churn berdasarkan atribut historisnya, sehingga tim CRM dan Marketing dapat melakukan intervensi retensi lebih awal?
3.	Bagaimana hasil prediksi model dapat diterjemahkan menjadi estimasi dampak bisnis (pendapatan yang berpotensi dipertahankan), sehingga dapat digunakan sebagai dasar keputusan program retensi?


**1.3 Goals**

> *Tujuan:* Menetapkan target yang ingin dicapai proyek ini, sebagai turunan langsung dari problem statement di atas.
Sebagai turunan langsung dari problem statement di atas, project ini bertujuan untuk:

1.	Mendukung penurunan churn rate dari kondisi historis 26,69% menuju mendekati benchmark industri telekomunikasi (±22%), melalui identifikasi pelanggan berisiko tinggi sejak dini.
2.	Menyediakan skor risiko churn (churn probability) per pelanggan yang dapat digunakan tim CRM dan Marketing untuk memprioritaskan dan menargetkan program retensi secara lebih efisien.
3.	Mengungkap faktor-faktor pendorong churn sebagai bahan evaluasi tim Product terhadap layanan, kontrak, atau fitur yang paling berkontribusi terhadap keputusan pelanggan untuk berhenti berlangganan.


**1.4 Analytical Approach**

> *Tujuan:* Menentukan pendekatan analitis/teknis (misal klasifikasi atau regresi) yang akan dipakai untuk mencapai goals.
Karena target (`Churn`) bersifat kategorikal biner (*Yes*/*No*), masalah ini didekati sebagai **binary classification**. Pendekatan yang akan dilakukan pada tahap-tahap selanjutnya:

•	Membangun dan membandingkan beberapa algoritma klasifikasi (Logistic Regression sebagai baseline yang mudah diinterpretasi, KNN, Decision Tree, Random Forest, AdaBoost, serta metode ensemble Voting/Stacking/Bagging) melalui 5-fold cross-validation.

•	Melakukan hyperparameter tuning pada kandidat model dengan performa terbaik dari tahap benchmarking.

•	Menangani ketidakseimbangan distribusi target (±27% churn vs ±73% tidak churn) menggunakan class_weight, atau oversampling/undersampling seperti SMOTE, sesuai mitigasi risiko teknis yang ditetapkan.

•	Menggunakan output probabilitas model sebagai churn risk score (bukan hanya label biner), sehingga dapat diperingkat oleh tim CRM/Marketing untuk menentukan prioritas tindak lanjut.

•	Melakukan interpretasi model menggunakan SHAP untuk menjawab Problem Statement mengenai faktor pendorong churn, sekaligus membangun kepercayaan stakeholder bisnis terhadap hasil model.

•	Model dievaluasi ulang (performance monitoring) setiap kuartal; retraining dilakukan apabila Recall atau ROC-AUC turun lebih dari 5% selama dua kuartal berturut-turut, atau terdeteksi perubahan distribusi data (data drift).


**1.5 Metric Evaluation (Business Metric, Machine Learning Evaluation Metric)**

> *Tujuan:* Menjembatani metrik bisnis (misal estimasi kerugian) dengan metrik ML (misal precision/recall) yang nanti dipakai mengevaluasi model.

**Business Metric**

Setiap pelanggan yang sebenarnya akan churn namun tidak terdeteksi oleh model (False Negative) berarti hilangnya kesempatan untuk menjalankan program retensi dan berpotensi kehilangan recurring revenue dari pelanggan tersebut. Sebaliknya, setiap pelanggan yang diprediksi churn padahal sebenarnya tidak (False Positive) menyebabkan biaya operasional retensi (follow-up oleh CRM/Marketing) yang sebetulnya tidak diperlukan.

Karena biaya kehilangan pelanggan (False Negative) pada umumnya jauh lebih besar dibandingkan biaya follow-up yang terbuang (False Positive) — mengingat biaya akuisisi pelanggan baru jauh lebih tinggi daripada biaya retensi — model dioptimalkan dengan memprioritaskan Recall, tanpa mengabaikan Precision, agar kapasitas dan anggaran tim retensi tetap terjaga.


**Machine Learning Evaluation Metric**

•	Recall (kelas Churn) dijadikan metrik utama, karena merepresentasikan seberapa banyak pelanggan yang benar-benar akan churn berhasil terdeteksi oleh model.

•	Precision tetap dipantau agar jumlah pelanggan yang menerima program retensi masih sesuai dengan kapasitas dan anggaran perusahaan.

•	ROC-AUC digunakan untuk melihat kemampuan model membedakan pelanggan churn dan tidak churn secara keseluruhan.

•	Baseline pembanding: model Logistic Regression sederhana, sebagai acuan minimal sebelum mempertimbangkan model yang lebih kompleks.

•	Akurasi tidak dijadikan metrik utama karena distribusi target tidak seimbang (±27:73), sehingga akurasi tinggi berpotensi menyesatkan (model bisa saja hanya menebak mayoritas kelas "tidak churn").


**1.6 Success Criteria**

> *Tujuan:* Menetapkan ambang batas angka (misal minimal recall 80%) yang menentukan apakah model dianggap layak dipakai.
Model dianggap layak digunakan apabila, pada data testing/unseen, mencapai:

•	Recall (kelas Churn) > 75%

•	ROC-AUC ≥ 0.80

Dari sisi bisnis, keberhasilan project diukur dari kontribusinya terhadap penurunan churn rate dari 26,69% menuju mendekati benchmark industri (±22%).

**Ilustrasi potensi dampak bisnis (simulasi)**
Simulasi berikut disusun untuk mengilustrasikan potensi dampak bisnis apabila program retensi berhasil menurunkan churn rate ke level benchmark industri — bukan hasil implementasi aktual.

•	Kondisi saat ini: churn rate 26,69% → 1.316 pelanggan churn (dari total 4.930 pelanggan).

•	Target: churn rate 22% → 1.085 pelanggan churn, sehingga sekitar 231 pelanggan berpotensi dipertahankan.

•	Dengan asumsi revenue Rp50.000 per pelanggan per bulan, potensi pendapatan yang dipertahankan diperkirakan sekitar Rp11.550.000 per bulan, atau sekitar Rp138.600.000 per tahun (asumsi pelanggan yang dipertahankan tetap berlangganan selama 12 bulan penuh).


> **Catatan:** Angka simulasi di atas bersifat ilustratif dan bergantung pada beberapa asumsi (revenue per pelanggan flat, retensi penuh 12 bulan, dan target churn rate mengacu pada benchmark industri). Angka ini sebaiknya dikonfirmasi ulang bersama tim Finance/CRM sebelum digunakan sebagai dasar keputusan bisnis.

## **Section 2. Data Understanding**

>   *Tujuan:* Mengenali data secara umum - bentuk, tipe, dan makna tiap fitur - sebelum melakukan pembersihan atau analisis mendalam.

**2.1 General Information**

>   *Tujuan:* Melihat ukuran, tipe data, dan struktur umum dataset (jumlah baris, kolom, tipe data tiap kolom).

In [4]:
DATA_PATH = "../data/raw/data_telco_customer_churn.csv"
df = pd.read_csv(DATA_PATH)

In [ ]:
df.head()

,Dependents,tenure,OnlineSecurity,OnlineBackup,InternetService,DeviceProtection,TechSupport,Contract,PaperlessBilling,MonthlyCharges,Churn
0,Yes,9,No,No,DSL,Yes,Yes,Month-to-month,Yes,72.90,Yes
1,No,14,No,Yes,Fiber optic,Yes,No,Month-to-month,Yes,82.65,No
2,No,64,Yes,No,DSL,Yes,Yes,Two year,No,47.85,Yes
3,No,72,Yes,Yes,DSL,Yes,Yes,Two year,No,69.65,No
4,No,3,No internet service,No internet service,No,No internet service,No internet service,Month-to-month,Yes,23.60,No


In [6]:
df.shape

(4930, 11)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4930 entries, 0 to 4929
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Dependents        4930 non-null   object 
 1   tenure            4930 non-null   int64  
 2   OnlineSecurity    4930 non-null   object 
 3   OnlineBackup      4930 non-null   object 
 4   InternetService   4930 non-null   object 
 5   DeviceProtection  4930 non-null   object 
 6   TechSupport       4930 non-null   object 
 7   Contract          4930 non-null   object 
 8   PaperlessBilling  4930 non-null   object 
 9   MonthlyCharges    4930 non-null   float64
 10  Churn             4930 non-null   object 
dtypes: float64(1), int64(1), object(9)
memory usage: 423.8+ KB


In [9]:
print(df.isnull().sum())

Dependents          0
tenure              0
OnlineSecurity      0
OnlineBackup        0
InternetService     0
DeviceProtection    0
TechSupport         0
Contract            0
PaperlessBilling    0
MonthlyCharges      0
Churn               0
dtype: int64


**Interpretasi:**

- **Tidak ditemukan missing value** pada seluruh kolom — setiap kolom memiliki 4.930 baris non-null, sama dengan total baris. Ini menyederhanakan tahap Data Cleaning karena tidak diperlukan strategi imputasi.
- Tipe data terbagi menjadi tiga kelompok: **numerik** (`tenure` bertipe `int64`, `MonthlyCharges` bertipe `float64`), dan **kategorikal/object** (9 kolom sisanya, termasuk target `Churn`).
- Karena mayoritas fitur berupa data kategorikal (object), tahap Data Preparation nanti perlu melibatkan proses *encoding* (mis. One-Hot Encoding) sebelum data bisa dipakai oleh algoritma machine learning.

**2.2 Feature Information**

>   *Tujuan:* Mendokumentasikan makna tiap fitur dan relevansinya terhadap masalah bisnis di Section 1.

| Feature | Type | Description | Impact to Business |
|---|---|---|---|
| `Dependents` | Kategorikal (Yes/No) | Status apakah pelanggan memiliki tanggungan | Pelanggan dengan tanggungan cenderung punya kebutuhan komunikasi/layanan yang lebih stabil, berpotensi memengaruhi loyalitas |
| `tenure` | Numerik (0-72 bulan) | Lama berlangganan pelanggan dalam satuan bulan | Indikator loyalitas langsung; pelanggan baru (tenure rendah) secara umum lebih rentan churn dibanding pelanggan lama |
| `OnlineSecurity` | Kategorikal (Yes/No/No internet service) | Status kepemilikan layanan keamanan online | Layanan tambahan yang bisa meningkatkan *stickiness* pelanggan terhadap provider |
| `OnlineBackup` | Kategorikal (Yes/No/No internet service) | Status kepemilikan layanan backup online | Sama seperti di atas -- layanan add-on yang berpotensi menurunkan risiko churn |
| `InternetService` | Kategorikal (DSL/Fiber optic/No) | Jenis layanan internet yang digunakan pelanggan | Jenis layanan berkaitan langsung dengan kepuasan & biaya; relevan untuk mengevaluasi kualitas layanan per jenis internet |
| `DeviceProtection` | Kategorikal (Yes/No/No internet service) | Status kepemilikan layanan proteksi perangkat | Layanan add-on yang berpotensi meningkatkan retensi pelanggan |
| `TechSupport` | Kategorikal (Yes/No/No internet service) | Status kepemilikan layanan dukungan teknis | Pelanggan tanpa dukungan teknis lebih mungkin mengalami masalah tak terselesaikan yang mendorong churn |
| `Contract` | Kategorikal (Month-to-month/One year/Two year) | Jenis kontrak berlangganan | Komitmen kontrak jangka panjang secara struktural menurunkan kemungkinan churn (biaya berpindah lebih tinggi) |
| `PaperlessBilling` | Kategorikal (Yes/No) | Status penggunaan tagihan tanpa kertas | Proxy dari preferensi/kenyamanan digital pelanggan, dapat berkorelasi dengan segmen demografis tertentu |
| `MonthlyCharges` | Numerik (18.8-118.65) | Besaran tagihan layanan per bulan | Biaya yang lebih tinggi dapat menjadi pemicu churn jika pelanggan merasa tidak sepadan dengan value yang diterima |
| `Churn` (target) | Kategorikal (Yes/No) | Status apakah pelanggan berhenti berlangganan | Variabel target yang ingin diprediksi oleh model |

**2.3 Statistics Summary**

>   *Tujuan:* Melihat ringkasan statistik deskriptif (mean, median, min-max, dsb) untuk menangkap gambaran awal distribusi data.

In [8]:
# df.describe(include='all')
df.describe(include='all')

,Dependents,tenure,OnlineSecurity,OnlineBackup,InternetService,DeviceProtection,TechSupport,Contract,PaperlessBilling,MonthlyCharges,Churn
count,4930,4930.000000,4930,4930,4930,4930,4930,4930,4930,4930.000000,4930
unique,2,NaN,3,3,3,3,3,3,2,NaN,2
top,No,NaN,No,No,Fiber optic,No,No,Month-to-month,Yes,NaN,No
freq,3446,NaN,2445,2172,2172,2186,2467,2721,2957,NaN,3614
mean,NaN,32.401217,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.883032,NaN
std,NaN,24.501193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.923960,NaN
min,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.800000,NaN
25%,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.050000,NaN
50%,NaN,29.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN
75%,NaN,55.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN


**Interpretasi:**

*Fitur numerik:*
- `tenure` memiliki rentang 0-72 bulan dengan rata-rata sekitar 32,4 bulan dan median 29 bulan. Sebaran cukup lebar (std sekitar 24,5), menunjukkan basis pelanggan mencakup baik pelanggan yang baru bergabung maupun yang sudah sangat lama berlangganan.
- `MonthlyCharges` berkisar 18,8-118,65 dengan rata-rata sekitar 64,9. Median (70,35) lebih tinggi dari mean, mengindikasikan sedikit kemencengan (*skew*) ke arah nilai tagihan yang lebih rendah.

*Fitur kategorikal:*
- `Contract`: kategori terbanyak adalah **Month-to-month** (2.721 dari 4.930 pelanggan, sekitar 55%) -- konsisten dengan temuan sebelumnya bahwa kontrak bulanan memiliki churn rate tertinggi (43,3%), sehingga porsi signifikan pelanggan berada pada segmen paling berisiko.
- `InternetService`: kategori terbanyak adalah **Fiber optic** (2.172 pelanggan) -- juga selaras dengan temuan bahwa segmen ini memiliki churn rate tertinggi di antara jenis layanan internet.
- `TechSupport` dan `OnlineSecurity`: mayoritas pelanggan (masing-masing 2.467 dan 2.445) **tidak** memiliki layanan tersebut -- berpotensi menjadi salah satu pendorong churn utama, sesuai temuan pada Data Understanding sebelumnya.
- `Churn`: dari 4.930 pelanggan, 3.614 berlabel "No" dan sisanya "Yes" -- mengonfirmasi kembali proporsi churn sekitar 26,9%, yang tergolong **imbalanced** dan perlu ditangani pada tahap modeling.

## **Section 3. Data Cleaning**

>   *Tujuan:* Memastikan data bebas dari masalah kualitas (missing value, duplikat, inkonsistensi) sebelum dipakai lebih lanjut.

**3.1 Missing Values**

>   *Tujuan:* Mengidentifikasi kolom yang punya data hilang dan menentukan strategi menanganinya (drop, imputasi, atau dibiarkan dengan alasan tertentu).

**3.2 Duplicated Values**

>   *Tujuan:* Mengecek baris data yang terduplikasi penuh maupun sebagian (misal duplikat berdasarkan ID).

**3.3 Data Consistency Check**
- Spelling errors / typo pada kategori
- Inkonsistensi kapitalisasi & format penulisan
- Whitespace tersembunyi

>   *Tujuan:* Menyeragamkan penulisan nilai kategorikal supaya kategori yang sebenarnya sama tidak terbaca sebagai kategori berbeda oleh model.

>   Jangan hanya cek typo. Kesalahan yang jauh lebih sering muncul di data nyata adalah inkonsistensi format, misalnya kategori yang sama tapi ditulis berbeda seperti `"Yes"`, `"yes"`, dan `"YES "` (dengan spasi tersembunyi) - ini dianggap 3 kategori berbeda oleh model kalau tidak dibersihkan.

**3.4 Identify Anomaly Values**
- Check Distribution (Numerical Variable)
- Check Cardinality (Categorical Variable)

>   *Tujuan:* Mendeteksi nilai yang secara statistik tidak wajar (outlier pada numerik, kategori dengan cardinality aneh pada kategorikal).

## **Section 4. Exploratory Data Analysis (EDA)**

>   *Tujuan:* Menggali pola dan hubungan dalam data training untuk membangun intuisi sebelum masuk ke tahap modeling.

**5.1 Univariate Analysis**
- Distribusi target
- Distribusi fitur numerik
- Distribusi fitur kategorikal

>   *Tujuan:* Memahami karakteristik tiap variabel secara individual, termasuk seberapa seimbang distribusi target.

**5.2 Bivariate Analysis (terhadap Target)**

>   *Tujuan:* Mencari pola hubungan antara tiap fitur dengan target, untuk menjawab langsung Problem Statement di Section 1.2.

>   Ini bagian paling penting untuk menjawab Problem Statement di Section 1.2 - cari pola antara tiap fitur dengan target, bukan sekadar plot tanpa insight.

**5.3 Correlation & Multicollinearity Check**

>   *Tujuan:* Mengecek hubungan antar fitur untuk mendeteksi multikolinearitas yang bisa mengganggu interpretasi model nanti.

**5.4 Multivariate / Interaction Analysis (opsional)**

>   *Tujuan:* Menelusuri interaksi antara beberapa fitur sekaligus untuk pola yang lebih kompleks dari yang bisa ditangkap analisis dua arah.

## **Section 5. Data Preparation**

>   *Tujuan:* Mengubah data mentah menjadi bentuk siap pakai untuk pemodelan (numerik, terskala, tanpa kategori yang belum di-encode).

**5.1 Initialization**
- Initialization function
- Define Feature and Target

>   *Tujuan:* Menyiapkan fungsi bantu dan mendefinisikan mana kolom fitur (X) dan target (y) sebelum transformasi dimulai.

**5.2 Constructing `Training` and `Testing` Data (from `Seen` Dataset)**

>   *Tujuan:* Membagi data `Seen` menjadi training dan testing untuk keperluan pengembangan dan evaluasi model.

**5.3 Handling Imbalanced Data (jika relevan)**

>   *Tujuan:* Menangani ketimpangan proporsi kelas target supaya model tidak bias ke kelas mayoritas.

>   Cek proporsi kelas target di Section 5.1. Kalau timpang (misal 90:10), pertimbangkan strategi seperti class_weight, SMOTE, atau undersampling - **tapi ingat, teknik resampling hanya boleh diterapkan pada data training**, tidak pernah pada data testing/unseen, supaya evaluasi tetap realistis.

**5.4 Data Transformation (Feature Engineering)**

>   *Tujuan:* Melakukan encoding, scaling, atau transformasi lain agar data sesuai kebutuhan algoritma yang dipakai.

**5.5 Feature Selection**

>   *Tujuan:* Memilih fitur yang paling relevan/berkontribusi untuk mengurangi noise dan risiko overfitting.

**5.6 Overview**

>   *Tujuan:* Merangkum hasil akhir data preparation (bentuk data final) sebelum masuk ke tahap Model Development.

## **Section 6. Model Development**

>   *Tujuan:* Membangun, membandingkan, dan menyempurnakan model machine learning menggunakan data.

**6.1 Initialization**
- Initialization Function
- Create Custom Metrics
- Define Cross-Validation Strategy
- Create a workflow of the experiment

>   *Tujuan:* Menyiapkan fungsi metrik custom dan strategi cross-validation yang dipakai konsisten di seluruh eksperimen model.

>   Tentukan strategi CV secara eksplisit (misal `StratifiedKFold` untuk klasifikasi dengan target tidak seimbang) dan simpan `RANDOM_STATE` yang sama dari Section 0. Ingat prinsip **CV-first**: bandingkan model lewat cross-validation dulu, baru evaluasi akhir di data testing - jangan sebaliknya.

**6.2 Developing the Model Pipeline**

>   *Tujuan:* Merangkai seluruh langkah preprocessing dan model ke dalam satu objek Pipeline yang konsisten dipakai ulang.

>   Gunakan `Pipeline`/`ColumnTransformer` dari scikit-learn supaya seluruh langkah preprocessing (imputasi, encoding, scaling) ikut ter-*fit* hanya pada data training di setiap fold - ini mencegah data leakage antara fold CV.

**6.3 Model Benchmarking (Comparing model base performance)**

>   *Tujuan:* Membandingkan performa dasar beberapa algoritma (tanpa tuning) untuk memilih kandidat terbaik yang layak dituning lebih lanjut.

**6.4 Tune Model**

>   *Tujuan:* Mengoptimalkan hyperparameter dari model kandidat terbaik hasil benchmarking untuk meningkatkan performa.

**6.5 Analyze Model**
- Evaluate model on data testing
- Confusion Matrix / Threshold Analysis (Classification) atau Residual Analysis (Regression)
- Learning Curve Inspection

>   *Tujuan:* Mengevaluasi performa model secara mendalam di luar satu angka metrik utama, termasuk mengecek tanda overfitting/underfitting.

**6.6 Model Calibration (Classification Only)**

>   *Tujuan:* Menyesuaikan output probabilitas model supaya lebih merepresentasikan kemungkinan sebenarnya, penting saat threshold dipakai untuk keputusan bisnis.

**6.7 Model Explanation and Interpretation**
- Feature Importance (Tree Based Model) / Coefficient Regression (Regression Based Model)
- SHAP Value identification
- Counterfactual Analysis

>   *Tujuan:* Menjelaskan bagaimana model mengambil keputusan - penting untuk membangun kepercayaan stakeholder bisnis terhadap model.

## **Section 7. Model Deployment**

>   *Tujuan:* Menyiapkan model terlatih agar bisa dipakai di luar notebook, lengkap dengan dokumentasi teknis yang diperlukan.

**7.1 Export Model (joblib/pickle)**

>   *Tujuan:* Menyimpan pipeline terlatih ke dalam file yang bisa dimuat ulang tanpa perlu melatih ulang dari awal.

>   Minimal, export pipeline lengkap (bukan cuma model) dengan `joblib.dump()` supaya preprocessing dan model tetap satu paket saat dipakai ulang.

**7.2 Deployment Checklist**
- Versi library yang digunakan
- Format input yang diharapkan model
- Cara memuat ulang pipeline

>   *Tujuan:* Mendokumentasikan hal teknis yang perlu diperhatikan tim lain saat model dipakai di lingkungan produksi.

## **Section 8. Model Implementation**

>   *Tujuan:* Menjelaskan cara pakai model di dunia nyata, batasannya, dan dampak bisnisnya lewat simulasi.

**8.1 How to implement the model?**

>   *Tujuan:* Menjelaskan langkah teknis memakai model untuk melakukan prediksi pada data baru.

**8.2 What are the limitations of the model?**

>   *Tujuan:* Mengakui batasan model secara jujur, termasuk skenario di mana prediksinya kurang bisa diandalkan.

**8.3 Business Calculation (Simulation using unseen data)**

>   *Tujuan:* Mensimulasikan dampak bisnis dari penggunaan model, memakai data `unseen` yang belum pernah dilihat selama proses modeling.

>   Ini saatnya `unseen` data dipakai. Kaitkan hasil simulasi dengan metrik bisnis yang kamu tetapkan di Section 1.5 - misalnya, hitung estimasi kerugian akibat False Negative vs biaya operasional akibat False Positive, sesuai threshold yang dipilih.

## **Section 9. Conclusion and Recommendation**

>   *Tujuan:* Merangkum keseluruhan proyek dan menerjemahkan hasil teknis kembali ke bahasa yang dipahami stakeholder bisnis.

**9.1 Conclusion**
- Conclusion (Model)
- Conclusion (Business)

>   *Tujuan:* Merangkum temuan utama dari sisi performa model dan sisi dampak bisnis, menjawab kembali Goals di Section 1.3.

**9.2 Recommendation**
- Recommendation (Model)
- Recommendation (Business)

>   *Tujuan:* Memberikan rekomendasi tindak lanjut konkret berdasarkan temuan proyek, baik dari sisi teknis maupun bisnis.